In [ ]:
import pandas as pd
import re
from tqdm.auto import tqdm
from googleapiclient.discovery import build

In [ ]:
YOUTUBE_API_KEY = "YOUR_KEY_HERE"
youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

In [ ]:
spotify_clean = pd.read_csv("../data/processed/spotify_cleaned.csv")
spotify_clean.head()

In [ ]:
def search_youtube(track_name, artist_name, max_results=5):
    query = f"{track_name} {artist_name} official audio"
    
    request = youtube.search().list(
        part="snippet",
        q=query,
        type="video",
        maxResults=max_results
    )
    response = request.execute()
    
    results = []
    for item in response.get("items", []):
        results.append({
            "video_id": item["id"]["videoId"],
            "video_title": item["snippet"]["title"],
            "channel_title": item["snippet"]["channelTitle"],
            "published_at": item["snippet"]["publishedAt"]
        })
    return results

In [ ]:
test_results = search_youtube("Hold On", "Chord Overstreet")
pd.DataFrame(test_results)

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"\[.*?\]", " ", text)
    text = re.sub(r"official video|official audio|lyrics|lyric video|audio", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def score_match(track_name, artist_name, video_title):
    target = normalize_text(f"{track_name} {artist_name}")
    candidate = normalize_text(video_title)
    
    score = 0
    for token in target.split():
        if token in candidate:
            score += 1
    return score

In [ ]:
sample_df = spotify_clean[["track_id", "track_name", "artists", "album_name"]].head(20).copy()

rows = []

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    best = get_best_match(row["track_name"], row["artists"])
    
    out = {
        "track_id": row["track_id"],
        "track_name": row["track_name"],
        "artists": row["artists"],
        "album_name": row["album_name"],
    }
    
    if best is not None:
        out.update(best)
        out["video_found"] = True
    else:
        out["video_found"] = False
    
    rows.append(out)

youtube_matches = pd.DataFrame(rows)
youtube_matches.head()

In [ ]:
youtube_matches.to_csv("../data/interim/youtube_matches_sample.csv", index=False)